# Indian Medicinal Leaves — Full Multi-Feature Classification Benchmark
#
Pipeline:
1. Data loading
2. Preprocessing
3. Shape features (contour geometry + Hu moments)
4. Color features (HSV histogram + color moments)
5. Texture features (GLCM + LBP + Gabor)
6. Vein/edge-structure features (simplified, efficient version)
7. Keypoint features (ORB + Bag-of-Visual-Words)
8. Train every possible combination of the 5 feature groups (31 combos)
   on 5 classifiers (Random Forest, LightGBM, SVM, Naive Bayes, KNN)
9. Report accuracy/precision/recall/F1 per run, then a final leaderboard
#
Paste each "# %%" block into its own Kaggle notebook cell in order.

In [1]:
# ============================================================
# CELL 1 — IMPORTS & CONFIG
# ============================================================
import os
import glob
import time
import itertools
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2

from skimage.morphology import skeletonize
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from skimage.filters import gabor
from scipy.ndimage import distance_transform_edt
from scipy.stats import skew

from multiprocessing import Pool, cpu_count

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import MiniBatchKMeans
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report)

from lightgbm import LGBMClassifier

# ---- Paths (as given) ----
DATASET_ROOT = ("/kaggle/input/datasets/aryashah2k/indian-medicinal-leaves-dataset/"
                 "Indian Medicinal Leaves Image Datasets/Medicinal Leaf dataset")
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Global knobs (tune for speed vs. accuracy) ----
IMG_SIZE = 256                  # resize target for every image
MAX_IMAGES_PER_CLASS = None     # set an int (e.g. 100) for a quick smoke-test run first
TEST_SIZE = 0.2                 # train/test split fraction
RANDOM_STATE = 42               # fixed seed -> same split used for every combo/classifier
ORB_N_FEATURES = 300            # ORB keypoints per image (cap)
BOVW_N_CLUSTERS = 50            # visual-word vocabulary size (= keypoint feature dimension)

print("Config loaded. Dataset root:")
print(DATASET_ROOT)

Config loaded. Dataset root:
/kaggle/input/datasets/aryashah2k/indian-medicinal-leaves-dataset/Indian Medicinal Leaves Image Datasets/Medicinal Leaf dataset


In [2]:
# ============================================================
# CELL 2 — DATA LOADING
# ============================================================
def collect_image_paths(root_dir, max_per_class=None):
    """
    Walks every species sub-folder under root_dir and returns a list of
    (image_path, label) tuples. The sub-folder name IS the class label
    (e.g. "Aloevera", "Neem", ...).
    """
    records = []
    class_dirs = sorted([d for d in glob.glob(os.path.join(root_dir, "*")) if os.path.isdir(d)])
    print(f"Found {len(class_dirs)} class folders.")
    for class_dir in class_dirs:
        label = os.path.basename(class_dir)
        paths = (glob.glob(os.path.join(class_dir, "*.jpg")) +
                  glob.glob(os.path.join(class_dir, "*.jpeg")) +
                  glob.glob(os.path.join(class_dir, "*.png")) +
                  glob.glob(os.path.join(class_dir, "*.JPG")))
        if max_per_class:
            paths = paths[:max_per_class]
        for p in paths:
            records.append((p, label))
    print(f"Total images collected: {len(records)}")
    return records


records = collect_image_paths(DATASET_ROOT, MAX_IMAGES_PER_CLASS)
if len(records) == 0:
    raise RuntimeError("No images found — double check DATASET_ROOT matches the actual Kaggle input path.")

image_paths = [r[0] for r in records]
raw_labels = [r[1] for r in records]

# Encode string labels ("Aloevera", ...) into integers (0, 1, 2, ...) for sklearn.
label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(raw_labels)
N = len(records)
print(f"Encoded {len(label_encoder.classes_)} classes. Total samples: {N}")

# Split ONCE at the index level (not per-feature) so every feature group and
# every combination uses the exact same train/test partition -> fair comparison.
all_indices = np.arange(N)
train_idx, test_idx = train_test_split(
    all_indices, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all
)
print(f"Train samples: {len(train_idx)}  |  Test samples: {len(test_idx)}")

Found 80 class folders.
Total images collected: 6900
Encoded 80 classes. Total samples: 6900
Train samples: 5520  |  Test samples: 1380


In [3]:
# ============================================================
# CELL 3 — PREPROCESSING
# ============================================================
def load_and_segment(img_path, size=IMG_SIZE):
    """
    Loads one image and produces everything every later feature-extractor needs:
      - bgr: resized color image (for color features)
      - gray_eq: resized, grayscale, contrast-enhanced (CLAHE) image
                 (edges/veins/keypoints are much cleaner after this)
      - mask: binary mask isolating the leaf from the background (Otsu threshold
              + keep-largest-connected-component, so stray specks are removed)
    Returns (None, None, None) if the image can't be read.
    """
    bgr = cv2.imread(img_path)
    if bgr is None:
        return None, None, None
    bgr = cv2.resize(bgr, (size, size), interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    # Otsu auto-threshold: leaf is usually darker than a plain/light background.
    _, otsu_mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(otsu_mask, cv2.MORPH_OPEN, kernel)   # remove small noise specks
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)       # fill small holes

    # Keep only the single largest connected blob = the leaf itself.
    num_labels, labels_im = cv2.connectedComponents(mask)
    if num_labels > 1:
        sizes = [(labels_im == i).sum() for i in range(1, num_labels)]
        largest_label = 1 + int(np.argmax(sizes))
        mask = np.uint8(labels_im == largest_label) * 255

    # CLAHE = local contrast boost -> faint veins/edges become easier to detect.
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_eq = clahe.apply(gray)

    return bgr, gray_eq, mask

In [4]:
# ============================================================
# CELL 4 — SHAPE FEATURES (Contour geometry + Hu moments)
# ============================================================
def extract_shape_features(mask):
    """
    Contour-geometry descriptors + the 7 Hu invariant moments.
    Hu moments are log-scaled because their raw values span enormous ranges
    (some are ~1e-10, others ~1) which would otherwise dominate distance-based
    classifiers (SVM/KNN) purely due to scale, not information content.
    Output length: 7 (geometry) + 7 (Hu) = 14
    """
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return np.zeros(14)

    cnt = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, True)
    x, y, w, h = cv2.boundingRect(cnt)
    aspect_ratio = w / h if h != 0 else 0
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)
    solidity = area / hull_area if hull_area != 0 else 0          # how "filled-in" vs. concave
    circularity = (4 * np.pi * area) / (perimeter ** 2) if perimeter != 0 else 0
    extent = area / (w * h) if (w * h) != 0 else 0                # area vs. bounding box area
    equiv_diameter = np.sqrt(4 * area / np.pi) if area > 0 else 0

    moments = cv2.moments(cnt)
    hu = cv2.HuMoments(moments).flatten()
    hu_log = -np.sign(hu) * np.log10(np.abs(hu) + 1e-30)          # log-scale, sign-preserving

    return np.concatenate([
        [area, perimeter, aspect_ratio, solidity, circularity, extent, equiv_diameter],
        hu_log
    ])


SHAPE_COLS = ["area", "perimeter", "aspect_ratio", "solidity", "circularity",
              "extent", "equiv_diameter"] + [f"hu_{i}" for i in range(7)]

In [5]:
# ============================================================
# CELL 5 — COLOR FEATURES (HSV histogram + color moments)
# ============================================================
def extract_color_features(bgr, mask, hist_bins=16):
    """
    Two complementary descriptions of leaf color, computed only over leaf
    pixels (mask), never the background:
      - HSV histogram: distribution shape of hue/saturation/value
      - Color moments: mean, std, skewness per channel (a compact 3-number
        summary of each channel's distribution)
    Output length: 3*hist_bins (histogram) + 3*3 (moments) = 48 + 9 = 57 (bins=16)
    """
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    mask_bool = mask > 0
    if mask_bool.sum() == 0:
        return np.zeros(3 * hist_bins + 9)

    hist_feats = []
    moment_feats = []
    ranges = [(0, 180), (0, 256), (0, 256)]  # H:0-180, S/V:0-255 in OpenCV
    for ch in range(3):
        channel_vals = hsv[:, :, ch][mask_bool].astype(np.float32)
        hist, _ = np.histogram(channel_vals, bins=hist_bins, range=ranges[ch], density=True)
        hist_feats.extend(hist)

        mean = channel_vals.mean()
        std = channel_vals.std()
        sk = skew(channel_vals) if channel_vals.std() > 1e-6 else 0.0
        moment_feats.extend([mean, std, sk])

    return np.array(hist_feats + moment_feats)


COLOR_HIST_BINS = 16
COLOR_COLS = ([f"h_hist_{i}" for i in range(COLOR_HIST_BINS)] +
              [f"s_hist_{i}" for i in range(COLOR_HIST_BINS)] +
              [f"v_hist_{i}" for i in range(COLOR_HIST_BINS)] +
              ["h_mean", "h_std", "h_skew", "s_mean", "s_std", "s_skew",
               "v_mean", "v_std", "v_skew"])

In [6]:
# ============================================================
# CELL 6 — TEXTURE FEATURES (GLCM + LBP + Gabor)
# ============================================================
def extract_texture_features(gray_eq, mask, lbp_bins=10):
    """
    Three complementary texture descriptors, each capturing a different
    notion of "surface pattern":
      - GLCM (Gray-Level Co-occurrence Matrix): how often pairs of gray
        levels occur next to each other -> contrast, homogeneity, energy, etc.
      - LBP (Local Binary Pattern): a per-pixel "texture code" histogram,
        robust to overall brightness changes.
      - Gabor filters: responses to a few orientation/frequency-tuned filters,
        good at picking up directional patterns (e.g. brushed/striated surfaces).
    Output length: 6 (GLCM) + lbp_bins (10) + 8*2 (Gabor mean/var) = 6+10+16 = 32
    """
    mask_bool = mask > 0
    if mask_bool.sum() == 0:
        return np.zeros(6 + lbp_bins + 16)

    # Restrict to a tight bounding box around the leaf so background pixels
    # (which would all be one flat gray level) don't distort GLCM/Gabor stats.
    ys, xs = np.where(mask_bool)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    roi = gray_eq[y0:y1, x0:x1]

    # --- GLCM ---
    glcm = graycomatrix(roi, distances=[1], angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
                         levels=256, symmetric=True, normed=True)
    glcm_feats = [
        graycoprops(glcm, "contrast").mean(),
        graycoprops(glcm, "dissimilarity").mean(),
        graycoprops(glcm, "homogeneity").mean(),
        graycoprops(glcm, "energy").mean(),
        graycoprops(glcm, "correlation").mean(),
        graycoprops(glcm, "ASM").mean(),
    ]

    # --- LBP ---
    lbp = local_binary_pattern(gray_eq, P=8, R=1, method="uniform")
    lbp_vals = lbp[mask_bool]
    lbp_hist, _ = np.histogram(lbp_vals, bins=lbp_bins, range=(0, 10), density=True)

    # --- Gabor filter bank: 4 orientations x 2 frequencies = 8 filters ---
    gabor_feats = []
    for theta in [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]:
        for freq in [0.1, 0.3]:
            filt_real, _ = gabor(gray_eq, frequency=freq, theta=theta)
            vals = filt_real[mask_bool]
            gabor_feats.extend([vals.mean(), vals.var()])

    return np.concatenate([glcm_feats, lbp_hist, gabor_feats])


LBP_BINS = 10
TEXTURE_COLS = (["glcm_contrast", "glcm_dissimilarity", "glcm_homogeneity",
                  "glcm_energy", "glcm_correlation", "glcm_ASM"] +
                 [f"lbp_bin_{i}" for i in range(LBP_BINS)] +
                 [f"gabor_{i}" for i in range(16)])

In [7]:
# ============================================================
# CELL 7 — VEIN/EDGE-STRUCTURE FEATURES (simplified, efficient version)
# ============================================================
def extract_vein_edge_features(gray_eq, mask, orientation_bins=8):
    """
    The lightweight vein/edge pipeline agreed on earlier:
      Canny edge density, skeleton vein density, branch/endpoint counts,
      mean/std vein thickness, and an 8-bin edge-orientation histogram.
    Output length: 6 + orientation_bins (8) = 14
    """
    mask_bool = mask > 0
    leaf_area = mask_bool.sum()
    if leaf_area == 0:
        return np.zeros(6 + orientation_bins)

    # Canny edge density (fraction of leaf pixels that are edges)
    edges = cv2.Canny(gray_eq, 50, 150)
    edges_in_leaf = edges.copy()
    edges_in_leaf[~mask_bool] = 0
    edge_density = edges_in_leaf.sum() / 255.0 / leaf_area

    # Skeletonize -> thin the leaf silhouette/vein pattern to 1-pixel-wide lines
    binary = mask_bool.astype(np.uint8)
    skeleton = skeletonize(binary).astype(np.uint8)
    skeleton_pixels = skeleton.sum()
    vein_density = skeleton_pixels / leaf_area

    # Branch points (>=3 skeleton neighbors) and endpoints (1 neighbor)
    branch_count, endpoint_count = 0, 0
    if skeleton_pixels > 0:
        kernel = np.array([[1, 1, 1], [1, 10, 1], [1, 1, 1]])
        conv = cv2.filter2D(skeleton, -1, kernel)
        neighbor_counts = conv[skeleton.astype(bool)] - 10
        endpoint_count = int((neighbor_counts == 1).sum())
        branch_count = int((neighbor_counts >= 3).sum())

    # Vein thickness via distance transform, sampled along the skeleton
    dist_map = distance_transform_edt(binary)
    if skeleton_pixels > 0:
        mean_thickness = dist_map[skeleton.astype(bool)].mean()
        std_thickness = dist_map[skeleton.astype(bool)].std()
    else:
        mean_thickness, std_thickness = 0.0, 0.0

    # Edge-orientation histogram (Sobel gradient direction within the leaf)
    sobelx = cv2.Sobel(gray_eq, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray_eq, cv2.CV_64F, 0, 1, ksize=3)
    angles = np.arctan2(sobely, sobelx)[mask_bool]
    orient_hist, _ = np.histogram(angles, bins=orientation_bins, range=(-np.pi, np.pi), density=True)

    return np.concatenate([
        [edge_density, vein_density, branch_count, endpoint_count, mean_thickness, std_thickness],
        orient_hist
    ])


VEIN_COLS = (["edge_density", "vein_density", "branch_count", "endpoint_count",
              "mean_thickness", "std_thickness"] + [f"edge_orient_bin_{i}" for i in range(8)])

In [8]:
# ============================================================
# CELL 8a — KEYPOINT FEATURES, PART 1: raw ORB descriptor extraction
# ============================================================
# ORB is used instead of SIFT because it's free, fast, and works well for a
# bag-of-visual-words setup at this dataset size — an important speed
# consideration since this feature alone requires per-image detection AND a
# clustering step across the whole training set.
orb_detector = cv2.ORB_create(nfeatures=ORB_N_FEATURES)


def extract_orb_descriptors(gray_eq, mask):
    """
    Returns the raw ORB descriptor matrix for one image (shape: n_keypoints x 32),
    restricted to keypoints found inside the leaf mask. Returns None if no
    keypoints were found (e.g. a blank/corrupt image) — handled downstream.
    """
    keypoints, descriptors = orb_detector.detectAndCompute(gray_eq, mask)
    return descriptors  # dtype uint8, shape (n_kp, 32) or None

In [9]:
# ============================================================
# CELL 8b — PHASE A: run ALL preprocessing + feature extraction (parallel)
# ============================================================
# One worker function per image computes shape/color/texture/vein features
# AND raw ORB descriptors in a single pass (avoids re-loading/re-segmenting
# the image once per feature group, which would be 5x slower).
def process_one_image(args):
    idx, path = args
    bgr, gray_eq, mask = load_and_segment(path)
    if bgr is None or mask is None or mask.sum() == 0:
        return (idx, None, None, None, None, None)

    shape_f = extract_shape_features(mask)
    color_f = extract_color_features(bgr, mask)
    texture_f = extract_texture_features(gray_eq, mask)
    vein_f = extract_vein_edge_features(gray_eq, mask)
    orb_desc = extract_orb_descriptors(gray_eq, mask)

    return (idx, shape_f, color_f, texture_f, vein_f, orb_desc)


print(f"Running feature extraction (Phase A) across {cpu_count()} cores for {N} images...")
start = time.time()
with Pool(processes=cpu_count()) as pool:
    phase_a_results = pool.map(process_one_image, list(enumerate(image_paths)))
print(f"Phase A done in {time.time() - start:.1f}s")

# Sort back into original index order (Pool.map preserves order, but be explicit/safe)
phase_a_results.sort(key=lambda r: r[0])

# Pre-allocate feature matrices; failed images become all-zero rows (and are
# also tracked in `valid_mask` so you can inspect/drop them if needed).
X_shape = np.zeros((N, len(SHAPE_COLS)))
X_color = np.zeros((N, len(COLOR_COLS)))
X_texture = np.zeros((N, len(TEXTURE_COLS)))
X_vein = np.zeros((N, len(VEIN_COLS)))
orb_descriptors_per_image = [None] * N
valid_mask = np.zeros(N, dtype=bool)

for idx, shape_f, color_f, texture_f, vein_f, orb_desc in phase_a_results:
    if shape_f is None:
        continue
    X_shape[idx] = shape_f
    X_color[idx] = color_f
    X_texture[idx] = texture_f
    X_vein[idx] = vein_f
    orb_descriptors_per_image[idx] = orb_desc
    valid_mask[idx] = True

print(f"Valid images: {valid_mask.sum()} / {N}  (failed reads/empty masks excluded)")

Running feature extraction (Phase A) across 4 cores for 6900 images...


Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: 441 extraneous bytes before marker 0xd9


Phase A done in 3116.0s
Valid images: 6900 / 6900  (failed reads/empty masks excluded)


In [10]:
# ============================================================
# CELL 8c — PHASE B: build BoVW vocabulary (TRAIN SET ONLY) + histogram features
# ============================================================
# CRITICAL: the visual-word vocabulary must be learned only from training
# images. If test-image descriptors influenced the vocabulary, test
# information would leak into "features" the model sees before training,
# inflating accuracy in a way that wouldn't hold on truly unseen leaves.
train_only_descriptors = [orb_descriptors_per_image[i] for i in train_idx
                           if orb_descriptors_per_image[i] is not None]
all_train_descriptors = np.vstack(train_only_descriptors).astype(np.float32)
print(f"Total ORB descriptors pooled from training images: {all_train_descriptors.shape[0]}")

print(f"Fitting BoVW vocabulary (k={BOVW_N_CLUSTERS}) with MiniBatchKMeans...")
start = time.time()
bovw_kmeans = MiniBatchKMeans(n_clusters=BOVW_N_CLUSTERS, random_state=RANDOM_STATE,
                               batch_size=1000, n_init=3)
bovw_kmeans.fit(all_train_descriptors)
print(f"Vocabulary built in {time.time() - start:.1f}s")


def descriptors_to_bovw_histogram(descriptors, kmeans_model, n_clusters):
    """
    Converts one image's variable-length ORB descriptor set into a FIXED-length
    histogram: assign every descriptor to its nearest visual word (cluster),
    then count how often each word occurs. This is what makes keypoint
    features (naturally variable-count) combinable with the fixed-length
    shape/color/texture/vein vectors.
    """
    if descriptors is None or len(descriptors) == 0:
        return np.zeros(n_clusters)
    words = kmeans_model.predict(descriptors.astype(np.float32))
    hist, _ = np.histogram(words, bins=n_clusters, range=(0, n_clusters))
    hist = hist.astype(np.float64)
    total = hist.sum()
    return hist / total if total > 0 else hist  # normalize -> comparable across images


X_keypoint = np.zeros((N, BOVW_N_CLUSTERS))
for i in range(N):
    X_keypoint[i] = descriptors_to_bovw_histogram(orb_descriptors_per_image[i], bovw_kmeans, BOVW_N_CLUSTERS)

KEYPOINT_COLS = [f"bovw_word_{i}" for i in range(BOVW_N_CLUSTERS)]
print("Keypoint (BoVW) feature matrix ready:", X_keypoint.shape)

Total ORB descriptors pooled from training images: 1088104
Fitting BoVW vocabulary (k=50) with MiniBatchKMeans...
Vocabulary built in 1.0s
Keypoint (BoVW) feature matrix ready: (6900, 50)


In [11]:
# ============================================================
# CELL 8d — Save the full combined raw feature table (for reuse/inspection)
# ============================================================
feature_groups_raw = {
    "shape": (X_shape, SHAPE_COLS),
    "color": (X_color, COLOR_COLS),
    "texture": (X_texture, TEXTURE_COLS),
    "vein": (X_vein, VEIN_COLS),
    "keypoint": (X_keypoint, KEYPOINT_COLS),
}

all_cols = []
all_arrays = []
for name, (arr, cols) in feature_groups_raw.items():
    all_cols.extend([f"{name}__{c}" for c in cols])
    all_arrays.append(arr)

full_df = pd.DataFrame(np.hstack(all_arrays), columns=all_cols)
full_df["label"] = raw_labels
full_df["is_valid"] = valid_mask
full_df.to_csv(os.path.join(OUTPUT_DIR, "all_features_raw.csv"), index=False)
print(f"Saved raw combined feature table -> {OUTPUT_DIR}/all_features_raw.csv  shape={full_df.shape}")

Saved raw combined feature table -> /kaggle/working/all_features_raw.csv  shape=(6900, 169)


In [12]:
# ============================================================
# CELL 9a — PER-GROUP SCALING (fit on train only, apply to train+test)
# ============================================================
# Each group is standardized independently BEFORE any combination is formed.
# This is what makes it valid to concatenate, e.g., "area" (can be tens of
# thousands) with a BoVW histogram bin (0-1) — after scaling, every group
# contributes on a comparable footing regardless of its original units/range.
scaled_groups = {}
for name, (arr, cols) in feature_groups_raw.items():
    scaler = StandardScaler()
    scaler.fit(arr[train_idx])                 # fit on TRAIN rows only — no leakage
    scaled_groups[name] = scaler.transform(arr)  # apply to all rows (train+test)

print("Per-group scaling complete. Groups:", list(scaled_groups.keys()))

Per-group scaling complete. Groups: ['shape', 'color', 'texture', 'vein', 'keypoint']


In [13]:
# ============================================================
# CELL 9b — DEFINE CLASSIFIERS
# ============================================================
def build_classifiers():
    """A fresh set of un-fitted classifier instances (called once per run)."""
    return {
        "Random Forest": RandomForestClassifier(
            n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
        "LightGBM": LGBMClassifier(
            n_estimators=200, random_state=RANDOM_STATE, verbosity=-1, n_jobs=-1),
        "SVM": SVC(kernel="rbf", C=10, gamma="scale", random_state=RANDOM_STATE),
        "Naive Bayes": GaussianNB(),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    }

In [14]:
# ============================================================
# CELL 9c — TRAIN EVERY FEATURE-GROUP COMBINATION x EVERY CLASSIFIER
# ============================================================
group_names = list(scaled_groups.keys())  # ["shape", "color", "texture", "vein", "keypoint"]

y_train, y_test = y_all[train_idx], y_all[test_idx]

results = []                 # summary rows -> becomes the leaderboard
predictions_store = {}       # (combo_str, model_name) -> y_pred, for the detailed report later

combo_counter = 0
total_combos = sum(1 for r in range(1, len(group_names) + 1)
                    for _ in itertools.combinations(group_names, r))
print(f"Total feature-group combinations to test: {total_combos} "
      f"(x{len(build_classifiers())} classifiers = {total_combos * len(build_classifiers())} runs)")

for r in range(1, len(group_names) + 1):                       # combo sizes 1..5
    for combo in itertools.combinations(group_names, r):       # all combos of that size
        combo_counter += 1
        combo_str = "+".join(combo)

        # Build the combined, already-scaled feature matrix for this combo
        X_train_combo = np.hstack([scaled_groups[g][train_idx] for g in combo])
        X_test_combo = np.hstack([scaled_groups[g][test_idx] for g in combo])

        print(f"\n[{combo_counter}/{total_combos}] Feature combo: {combo_str}  "
              f"(dim={X_train_combo.shape[1]})")

        classifiers = build_classifiers()
        for model_name, model in classifiers.items():
            start = time.time()
            model.fit(X_train_combo, y_train)
            y_pred = model.predict(X_test_combo)
            elapsed = time.time() - start

            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
            rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
            f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

            # Concise one-line report per run (full 50-class reports are saved
            # only for the eventual best combo, see final cell — 155 full
            # reports would be unreadable).
            print(f"    {model_name:15s} | Acc: {acc:.4f} | Prec(macro): {prec:.4f} | "
                  f"Rec(macro): {rec:.4f} | F1(macro): {f1:.4f} | {elapsed:.1f}s")

            results.append({
                "combo": combo_str,
                "n_groups": r,
                "n_features": X_train_combo.shape[1],
                "classifier": model_name,
                "accuracy": acc,
                "precision_macro": prec,
                "recall_macro": rec,
                "f1_macro": f1,
                "train_time_sec": elapsed,
            })
            predictions_store[(combo_str, model_name)] = y_pred

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(OUTPUT_DIR, "all_combo_results.csv"), index=False)
print(f"\nAll runs complete. Full results saved -> {OUTPUT_DIR}/all_combo_results.csv")

Total feature-group combinations to test: 31 (x5 classifiers = 155 runs)

[1/31] Feature combo: shape  (dim=14)
    Random Forest   | Acc: 0.2572 | Prec(macro): 0.2933 | Rec(macro): 0.2680 | F1(macro): 0.2619 | 4.1s
    LightGBM        | Acc: 0.0319 | Prec(macro): 0.0229 | Rec(macro): 0.0315 | F1(macro): 0.0247 | 5.9s
    SVM             | Acc: 0.1696 | Prec(macro): 0.1784 | Rec(macro): 0.1666 | F1(macro): 0.1530 | 3.6s
    Naive Bayes     | Acc: 0.0761 | Prec(macro): 0.0663 | Rec(macro): 0.0984 | F1(macro): 0.0560 | 0.0s
    KNN             | Acc: 0.1384 | Prec(macro): 0.1552 | Rec(macro): 0.1406 | F1(macro): 0.1319 | 0.1s

[2/31] Feature combo: color  (dim=57)
    Random Forest   | Acc: 0.5848 | Prec(macro): 0.6114 | Rec(macro): 0.5853 | F1(macro): 0.5830 | 6.7s
    LightGBM        | Acc: 0.5304 | Prec(macro): 0.5366 | Rec(macro): 0.5249 | F1(macro): 0.5210 | 27.4s
    SVM             | Acc: 0.4993 | Prec(macro): 0.5131 | Rec(macro): 0.4956 | F1(macro): 0.4946 | 4.8s
    Naive Bayes 

In [15]:
# ============================================================
# CELL 10 — FINAL LEADERBOARD & BEST FEATURE COMBINATION
# ============================================================
# Rank primarily by F1 (macro) since it balances precision and recall across
# all ~50 classes, rather than accuracy alone which can hide poor performance
# on smaller/harder classes.
leaderboard = results_df.sort_values("f1_macro", ascending=False).reset_index(drop=True)

print("=" * 90)
print("TOP 15 (feature combination x classifier) BY MACRO F1-SCORE")
print("=" * 90)
print(leaderboard.head(15)[["combo", "classifier", "accuracy", "precision_macro",
                             "recall_macro", "f1_macro", "n_features"]].to_string(index=False))

# Best classifier PER combo-size, so you can see whether adding more feature
# groups actually helps or just adds noise/overfitting risk.
print("\n" + "=" * 90)
print("BEST RESULT AT EACH COMBINATION SIZE (1 to 5 feature groups)")
print("=" * 90)
best_per_size = leaderboard.loc[leaderboard.groupby("n_groups")["f1_macro"].idxmax()]
print(best_per_size[["n_groups", "combo", "classifier", "accuracy", "f1_macro"]].to_string(index=False))

# The single best (combo, classifier) pair overall
best_row = leaderboard.iloc[0]
best_combo, best_model = best_row["combo"], best_row["classifier"]
print("\n" + "=" * 90)
print(f"OVERALL BEST: {best_combo}  +  {best_model}")
print(f"Accuracy={best_row['accuracy']:.4f}  Precision(macro)={best_row['precision_macro']:.4f}  "
      f"Recall(macro)={best_row['recall_macro']:.4f}  F1(macro)={best_row['f1_macro']:.4f}")
print("=" * 90)

# Full, detailed per-class report — generated ONLY for the winning combination,
# using the predictions already stored (no need to refit).
best_preds = predictions_store[(best_combo, best_model)]
print("\nDetailed classification report for the best combination:\n")
print(classification_report(y_test, best_preds, target_names=label_encoder.classes_, zero_division=0))

leaderboard.to_csv(os.path.join(OUTPUT_DIR, "leaderboard_sorted.csv"), index=False)
print(f"\nSorted leaderboard saved -> {OUTPUT_DIR}/leaderboard_sorted.csv")

TOP 15 (feature combination x classifier) BY MACRO F1-SCORE
                            combo    classifier  accuracy  precision_macro  recall_macro  f1_macro  n_features
         shape+color+texture+vein           SVM  0.684783         0.709180      0.691573  0.691898         117
              shape+color+texture           SVM  0.672464         0.700088      0.683521  0.684276         103
               color+texture+vein           SVM  0.676812         0.696521      0.683301  0.682474         103
                    color+texture           SVM  0.678986         0.696371      0.682958  0.681704          89
               color+texture+vein Random Forest  0.665217         0.715614      0.672735  0.679234         103
              shape+color+texture Random Forest  0.665217         0.710310      0.675420  0.678527         103
         shape+color+texture+vein Random Forest  0.667391         0.717251      0.671453  0.677440         117
                    color+texture Random Forest  0.6